This is carcass removal data from various counties in Virginia. Hosted here: https://www.virginiaroads.org/datasets/ddc33a1134ca48f5874dc0f351f22db7_0/about


>The layer does not represent a complete dataset for all roads in Virginia. The number of Districts with maintenance contractors that participate in this form of data collection may increase over time. Data collection began in VDOT’s Culpeper and Staunton Districts in 2021. Time periods for which data are collected are not always continuous. There are occasional periods in which no data was collected, typically during transitions from one contract maintenance force to another.

>Wildlife carcass removals in the dataset were identified in two ways: (1) contract maintenance staff received a notification from VDOT requesting a carcass removal and (2) contract crews observed a carcass on or along the roadside while driving a routine route for maintenance activities.

Our goal is to standardize the data and to add more granular date information to support filtering in the Web Map

In [1]:
import pandas as pd
import numpy as np  
import re

In [2]:
# This is the original data from the VDOT Carcass Removal Map
Wildlifecarcass = pd.read_csv('Wildlife_Carcass_Removal_Tracking_5676419084963296149.csv')

In [3]:
# Inspect the data
Wildlifecarcass.head()


,ObjectID,GlobalID,Date and Pickup Time,Deer and Bear Removal,Other wildlife types,Route Name and Direction,Latitude_d,Longitude_d,County,District,Residency,x,y
0,83,ac10868c-9c94-444f-bb95-36b03bfd84a0,5/18/2021 12:43:01 PM,Deer,NaN,I-81S,37.835462,-79.370354,Rockbridge,Staunton,Lexington,-79.370354,37.835462
1,84,f529067b-0a14-401a-b0f7-b1701e771f4a,5/18/2021 2:15:55 PM,NaN,Coyote,I-81S,37.713655,-79.450178,Rockbridge,Staunton,Lexington,-79.450178,37.713655
2,85,d966d639-1944-4973-ac57-ad782bca5f38,5/18/2021 2:23:35 PM,NaN,Groundhog,I-81S,37.714697,-79.448655,Rockbridge,Staunton,Lexington,-79.448655,37.714697
3,86,7f5ea06a-c97d-4947-8e9e-aa5a14ca919f,5/18/2021 2:52:00 PM,Deer,NaN,I-81S,37.641463,-79.561258,Rockbridge,Staunton,Lexington,-79.561258,37.641463
4,88,2706ba75-9f2c-49a6-8eee-a7a4d02b42f1,5/18/2021 3:33:52 PM,Deer,NaN,I-81N,37.758094,-79.408782,Rockbridge,Staunton,Lexington,-79.408782,37.758094


## Create more granular date information columns

In [4]:
# With the date and pickup time column
Wildlifecarcass['Date'] = pd.to_datetime(Wildlifecarcass['Date and Pickup Time']).dt.date

In [5]:
Wildlifecarcass['Year'] = pd.to_datetime(Wildlifecarcass['Date and Pickup Time']).dt.year

In [6]:
Wildlifecarcass['MonthName'] = pd.to_datetime(Wildlifecarcass['Date and Pickup Time']).dt.month_name()

In [7]:
Wildlifecarcass['MonthNumber'] =pd.to_datetime(Wildlifecarcass['Date and Pickup Time']).dt.month

In [8]:
Wildlifecarcass['DOWName'] = pd.to_datetime(Wildlifecarcass['Date and Pickup Time']).dt.day_name()

In [9]:
Wildlifecarcass['DOWNumber'] = pd.to_datetime(Wildlifecarcass['Date and Pickup Time']).dt.day_of_week

## Merge the Species Columns

In [10]:
# make a new column named 'wildlife'
Wildlifecarcass['wildlife'] = Wildlifecarcass['Deer and Bear Removal']

In [11]:
# What are the unique values in the 'wildlife' column?
Wildlifecarcass['wildlife'].unique()

array(['Deer', nan, 'Bear', 'Other', 'Notfound'], dtype=object)

In [12]:
# for the rows where the 'wildlife' column is in Nan, 'Other', 'Notfound' fill it with the value from the 'Other wildlife types' column
Wildlifecarcass.loc[Wildlifecarcass['wildlife'].isin([np.nan, 'Other', 'Notfound']), 'wildlife'] = Wildlifecarcass['Other wildlife types']

In [13]:
# Check what the unique species values are in the new 'wildlife' column
Wildlifecarcass['wildlife'].unique()

array(['Deer', 'Coyote', 'Groundhog', 'Skunk', 'Bear', 'Raccoon',
       'Otheranimal', 'Dog', 'Opossum', 'Bird', 'Cat', 'Otheranimal,Bird',
       'Fox', nan, 'Bobcat', 'Farmanimal', 'Squirrel', 'Turtle'],
      dtype=object)

In [14]:
# we find that the following values are questionable and warrent further investigation
wildlife_values_to_check = ['Otheranimal', 'Otheranimal,Bird', 'Farmanimal', 'Other', 'Notfound', np.nan]
# print rows from the dataframe where the 'wildlife' column is in the list of values to check
Wildlifecarcass[Wildlifecarcass['wildlife'].isin(wildlife_values_to_check)]

,ObjectID,GlobalID,Date and Pickup Time,Deer and Bear Removal,Other wildlife types,Route Name and Direction,Latitude_d,Longitude_d,County,District,Residency,x,y,Date,Year,MonthName,MonthNumber,DOWName,DOWNumber,wildlife
24,114,7846beb8-0768-4160-bcf1-3094b8213944,5/25/2021 1:55:34 PM,NaN,Otheranimal,I-81N,38.241979,-78.961527,Augusta,Staunton,Harrisonburg,-78.961527,38.241979,2021-05-25,2021,May,5,Tuesday,1,Otheranimal
86,215,543e8b37-b75a-4d6f-ad06-48e837ef7ce6,6/21/2021 4:03:20 PM,NaN,Otheranimal,I-81N,37.973772,-79.189059,Augusta,Staunton,Harrisonburg,-79.189059,37.973772,2021-06-21,2021,June,6,Monday,0,Otheranimal
88,217,83275414-53c9-4bf4-b0f3-d367871ddeb9,6/21/2021 5:23:51 PM,NaN,Otheranimal,I-81N,37.664289,-79.519778,Rockbridge,Staunton,Lexington,-79.519778,37.664289,2021-06-21,2021,June,6,Monday,0,Otheranimal
106,257,4f812eac-373c-4c68-84c1-fd036a0ff582,6/25/2021 6:48:05 PM,NaN,"Otheranimal,Bird",I-64W,38.031154,-78.596384,Albemarle,Culpeper,Charlottesville,-78.596384,38.031154,2021-06-25,2021,June,6,Friday,4,"Otheranimal,Bird"
204,362,b002c3da-d8a9-4a41-9e85-a68c6868524b,7/21/2021 6:45:15 PM,NaN,Otheranimal,I-81N,39.250782,-78.105115,Frederick,Staunton,Edinburg,-78.105115,39.250782,2021-07-21,2021,July,7,Wednesday,2,Otheranimal
226,384,4f98c27c-b857-4473-98ee-3121baf3b83f,7/26/2021 7:41:24 PM,NaN,Otheranimal,I-66w,39.011794,-78.328908,Shenandoah,Staunton,Edinburg,-78.328908,39.011794,2021-07-26,2021,July,7,Monday,0,Otheranimal
240,398,a0235fce-8b8c-4e0b-bf5f-9c3dbb0dc3fd,7/28/2021 6:19:39 PM,NaN,Otheranimal,I—66 W,38.896552,-77.914484,Fauguier,Culpeper,Warrenton,-77.914484,38.896552,2021-07-28,2021,July,7,Wednesday,2,Otheranimal
253,417,8182bc4e-365f-4083-855f-7a4c104a6939,8/2/2021 2:38:14 PM,NaN,NaN,NaN,38.035827,-78.696457,Albemarle,Culpeper,Charlottesville,-78.696457,38.035827,2021-08-02,2021,August,8,Monday,0,NaN
283,449,56e77acb-d4bf-4f91-a2bb-669e4a8b9ff3,8/12/2021 12:38:11 PM,NaN,Otheranimal,I-64E,38.013576,-78.467338,Albemarle,Culpeper,Charlottesville,-78.467338,38.013576,2021-08-12,2021,August,8,Thursday,3,Otheranimal
291,480,03062749-5e4f-4de3-9dce-1be0fab7c402,8/18/2021 4:48:39 PM,NaN,Otheranimal,I-81S,38.002394,-79.173351,Augusta,Staunton,Harrisonburg,-79.173351,38.002394,2021-08-18,2021,August,8,Wednesday,2,Otheranimal


In [15]:
# We drop these rows from the dataframe, after inspection it is clear that there is no way to determine species for these rows
Wildlifecarcass = Wildlifecarcass.drop(Wildlifecarcass[Wildlifecarcass['wildlife'].isin(wildlife_values_to_check)].index)

In [16]:
# Check Value Counts of the 'wildlife' column
Wildlifecarcass.value_counts('wildlife')

wildlife
Deer         2363
Bear          196
Coyote         56
Raccoon        49
Dog            19
Opossum        18
Fox            18
Bird           12
Groundhog      11
Cat             9
Bobcat          8
Skunk           7
Squirrel        5
Turtle          2
Name: count, dtype: int64

## Standardizing the Route strings

In [17]:
#inspect the data frames 
Wildlifecarcass.head()

,ObjectID,GlobalID,Date and Pickup Time,Deer and Bear Removal,Other wildlife types,Route Name and Direction,Latitude_d,Longitude_d,County,District,Residency,x,y,Date,Year,MonthName,MonthNumber,DOWName,DOWNumber,wildlife
0,83,ac10868c-9c94-444f-bb95-36b03bfd84a0,5/18/2021 12:43:01 PM,Deer,NaN,I-81S,37.835462,-79.370354,Rockbridge,Staunton,Lexington,-79.370354,37.835462,2021-05-18,2021,May,5,Tuesday,1,Deer
1,84,f529067b-0a14-401a-b0f7-b1701e771f4a,5/18/2021 2:15:55 PM,NaN,Coyote,I-81S,37.713655,-79.450178,Rockbridge,Staunton,Lexington,-79.450178,37.713655,2021-05-18,2021,May,5,Tuesday,1,Coyote
2,85,d966d639-1944-4973-ac57-ad782bca5f38,5/18/2021 2:23:35 PM,NaN,Groundhog,I-81S,37.714697,-79.448655,Rockbridge,Staunton,Lexington,-79.448655,37.714697,2021-05-18,2021,May,5,Tuesday,1,Groundhog
3,86,7f5ea06a-c97d-4947-8e9e-aa5a14ca919f,5/18/2021 2:52:00 PM,Deer,NaN,I-81S,37.641463,-79.561258,Rockbridge,Staunton,Lexington,-79.561258,37.641463,2021-05-18,2021,May,5,Tuesday,1,Deer
4,88,2706ba75-9f2c-49a6-8eee-a7a4d02b42f1,5/18/2021 3:33:52 PM,Deer,NaN,I-81N,37.758094,-79.408782,Rockbridge,Staunton,Lexington,-79.408782,37.758094,2021-05-18,2021,May,5,Tuesday,1,Deer


In [18]:
# Change the column named Route Name and Direction to 'Route'
Wildlifecarcass = Wildlifecarcass.rename(columns={'Route Name and Direction': 'Route'})

In [19]:
# Find the names for the routes currently in the dataset
Wildlifecarcass['Route'].unique()

array(['I-81S', 'I-81N', 'I-64E', 'I-64W', nan, 'I-64 E', 'I 64E',
       'I-64 W', 'I 64 w', 'I-81 w', 'I 66 east', 'I-66 east', 'I-81 N',
       'I81 N', 'I-66 W', 'I-66E', 'I-81 S', 'I-66 west', 'I-66 WB ',
       'I-66 EB', 'I-66 E', 'I-66w', 'I-81 W', '1-81 W', 'I66 E',
       'I-64 east ', '64w', 'I-66', 'I66', 'I-66 ', 'W', 'I64w', 'I64E',
       'EAST', '64E', 'WBRS', 'I64EB', 'I64W', 'I-64EAST', 'I-64WEST',
       'I64WESTB', 'I-66W', 'I-81s'], dtype=object)

In [20]:
# Standardize the Route Name
Wildlifecarcass['Route'] = Wildlifecarcass['Route'].str.upper()
# Remove all whitespace
Wildlifecarcass['Route'] = Wildlifecarcass['Route'].str.replace(r'\s+', '', regex=True)
# Remove all characters except I, numbers, and hyphens. We don't care about east/west/north/south designations
Wildlifecarcass['Route'] = Wildlifecarcass['Route'].str.replace(r'[^I0-9-]', '', regex=True)

# Function to standardize routes
def standardize_route(route):
    if isinstance(route, str) and route:  # Check if the input is a non-empty string
        # Handle specific case for '1-<number>'
        if re.match(r'1-(\d+)', route):
            return f'I-{route.split("-")[1]}'
        
        # Handle cases like I-<number>, I<number>, and <number>
        match = re.match(r'^(I-?(\d+)|(\d+))$', route)  # Matches I-<number>, I<number>, or <number>
        if match:
            # Return as I-<number>
            return f'I-{match.group(2) or match.group(3)}'
    
    return np.nan  # Return NaN for non-string inputs or unmatched patterns

# Apply the standardization function
Wildlifecarcass['Standardized route'] = Wildlifecarcass['Route'].apply(standardize_route)

In [21]:
# Checking unique values to see what transformations were made
Wildlifecarcass['Route'].unique()

array(['I-81', 'I-64', nan, 'I64', 'I66', 'I-66', 'I81', '1-81', '64', ''],
      dtype=object)

In [22]:
# Double check to see if any originial routes got mislabled as NaN
mask = Wildlifecarcass['Route'].notna() & Wildlifecarcass['Standardized route'].isna()
result = Wildlifecarcass[mask]
result[["Route", "Standardized route"]]
# Looks like all of these are just an empty string and were correctly labeled as NaN


,Route,Standardized route
272,,NaN
308,,NaN
321,,NaN
332,,NaN


In [23]:
# Checking unique values of the new standardized route column
Wildlifecarcass['Standardized route'].value_counts(dropna = False)

Standardized route
I-64    1107
NaN      970
I-81     412
I-66     284
Name: count, dtype: int64

In [24]:
# Counting up the routes, NaN is the second highest 
Wildlifecarcass['Standardized route'].value_counts(dropna = False)

Standardized route
I-64    1107
NaN      970
I-81     412
I-66     284
Name: count, dtype: int64

In [25]:
# Export the cleaned dataframe to a new csv
Wildlifecarcass.to_csv('Wildlife_Carcass_Removal_Tracking_Cleaned.csv', index=False)